# 23 · Self-RAG / Corrective-RAG（自评 + 自纠循环）

> **学习目标**：实现一个最小可跑的「检索 → 自评 → 不行就再检 / 拒答」循环。理解 Self-RAG 与 Corrective-RAG (CRAG) 的核心思想，**不依赖任何特殊训练的模型**。
>
> **预备**：16、22 跑过。
>
> **为什么重要**：朴素 RAG 一次召回 → 一次生成，错了就错。**真实生产**需要 RAG 自己能判断「这次召回够不够」「答案对不对」，**自动多轮**修正。这也是 Agent 思想在 RAG 里的渗透。

In [ ]:
MODE = 'OFFLINE'

import numpy as np, hashlib, re, requests, json
OLLAMA = 'http://127.0.0.1:11434'

def fake_embed(text, dim=256):
    seed = int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)
    v = np.random.default_rng(seed).standard_normal(dim).astype(np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_embed(text):
    r = requests.post(f'{OLLAMA}/api/embeddings', json={'model':'nomic-embed-text','prompt':text}, timeout=30)
    r.raise_for_status()
    v = np.array(r.json()['embedding'], dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

def ollama_chat(prompt, model='qwen1.5_1.8', temp=0.0):
    r = requests.post(f'{OLLAMA}/api/chat', json={
        'model': model, 'stream': False, 'options': {'temperature': temp},
        'messages':[{'role':'user','content':prompt}]
    }, timeout=120)
    r.raise_for_status()
    return r.json()['message']['content']

if MODE == 'ONLINE':
    try: requests.get(f'{OLLAMA}/api/tags', timeout=1).raise_for_status(); embed = ollama_embed; chat = ollama_chat; print('✅ ONLINE')
    except Exception: MODE='OFFLINE'
if MODE == 'OFFLINE':
    embed = fake_embed
    # 见下面具体 stub
    print('OFFLINE 模式：用规则模拟 LLM 评估 / 生成')

## 1. Self-RAG 的核心循环

**简化版本流程**：

```
1. retrieve(query) → context
2. is_sufficient(query, context) ─── YES ──▶ generate(query, context)
         │                                          │
         NO                                         ▼
         │                                       answer
         ▼
3. rewrite_query / expand_context / fail
         │
         └─── 回到 1，最多 N 轮，超出则拒答
```

**关键约束**：必须有**循环上限**（如 max_iter=3），否则可能死循环。

In [ ]:
# 准备语料
DOCS = [
    '锂电池 XZ4054H 容量 300mAh，循环寿命 500 次。',
    'XZ5352R 是 800mAh 锂电池。',
    'XZ4054H-NE1.11 集成保护电路，是 XZ4054H 的升级版。',
    'PM8916 是电源管理芯片，工作电压 3.3V。',
    'Transformer 由 Google 2017 年提出。',
    'RAG 把外部知识接进 LLM 上下文。',
]
doc_vecs = np.vstack([embed(d) for d in DOCS])

def retrieve(query, top_k=3):
    q = embed(query)
    sims = doc_vecs @ q
    return [(int(i), float(sims[i]), DOCS[i]) for i in np.argsort(-sims)[:top_k]]

## 2. 自评：context 够不够？

**关键 prompt 设计**：
- 让 LLM **只输出 JSON**（避免长篇大论解析麻烦）
- 给 3 档：`sufficient` / `partial` / `insufficient`
- 附「为什么」帮助你 debug，但**程序只用 verdict**

In [ ]:
EVAL_PROMPT = '''你是一个严谨的评估员。给定用户问题和检索到的上下文，判断上下文是否足以回答问题。
只输出 JSON，格式 {{"verdict": "sufficient|partial|insufficient", "reason": "..."}}

问题：{question}
上下文：
{context}

判断（JSON）：'''

def parse_verdict(text: str) -> dict:
    """鲁棒地从 LLM 输出里抠出 JSON"""
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except json.JSONDecodeError: pass
    return {'verdict': 'partial', 'reason': 'parse_failed'}

def evaluate_context(question: str, context: str) -> dict:
    if MODE == 'ONLINE':
        resp = chat(EVAL_PROMPT.format(question=question, context=context))
        return parse_verdict(resp)
    # OFFLINE stub: 用 token 重叠判定
    q_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', question.lower()))
    c_tokens = set(re.findall(r'[\u4e00-\u9fff]|[A-Za-z0-9]+', context.lower()))
    overlap = len(q_tokens & c_tokens)
    if overlap >= 4:    return {'verdict': 'sufficient',   'reason': f'overlap={overlap}'}
    if overlap >= 2:    return {'verdict': 'partial',      'reason': f'overlap={overlap}'}
    return                   {'verdict': 'insufficient',  'reason': f'overlap={overlap}'}

# Smoke
for q in ['XZ4054H 容量', '今天天气怎么样', '锂电池升级']:
    hits = retrieve(q, top_k=2)
    ctx = '\n'.join(h[2] for h in hits)
    print(f'\nQ: {q!r}\n  ctx: {ctx[:80]}\n  评估: {evaluate_context(q, ctx)}')

## 3. 改写：当 context 不够时怎么换 query

**两种改写策略**：
- **扩展**：给 query 加同义词 / 上位概念（适合「query 太具体没召回」）
- **分解**：把复杂 query 拆成 2-3 个简单 sub-query 各查一遍（适合「query 是组合问题」）

本 notebook 用最简单的「补充关键词」策略演示。

In [ ]:
REWRITE_PROMPT = '''原 query 检索结果不够。请改写成一个更宽泛、更易匹配文档的版本。
只输出改写后的 query 一行，不要解释。

原 query: {query}
已检索到的不充分上下文（避免再选到这些）:
{context}

改写后:'''

def rewrite_query(query: str, failed_context: str) -> str:
    if MODE == 'ONLINE':
        return chat(REWRITE_PROMPT.format(query=query, context=failed_context)).strip().split('\n')[0]
    # OFFLINE stub：加一个常用通用词
    suffixes = [' 规格', ' 概念', ' 是什么', ' 详细介绍']
    for s in suffixes:
        if not query.endswith(s):
            return query + s
    return query + ' 更多信息'

## 4. 生成（带「拒答」选项）

In [ ]:
GEN_PROMPT = '''你是一个严谨的助手。仅基于「上下文」回答问题。
若上下文不足以回答，必须明确回复「根据已有资料，我无法回答这个问题」。
不要编造任何上下文里没有的信息。

上下文：
{context}

问题：{question}
回答：'''

def generate(question: str, context: str) -> str:
    if MODE == 'ONLINE':
        return chat(GEN_PROMPT.format(question=question, context=context))
    # OFFLINE stub：拼成「基于上下文」的简易回答
    if not context.strip():
        return '根据已有资料，我无法回答这个问题'
    return f'[STUB-LLM] 基于上下文：{context[:100]}... 我能告诉你：{question}'

## 5. 组装 Self-RAG 循环

In [ ]:
def self_rag(question: str, max_iter: int = 3, top_k: int = 3, verbose: bool = True) -> dict:
    trace = []
    current_query = question
    all_retrieved_ids = set()

    for it in range(max_iter):
        # 检索
        hits = retrieve(current_query, top_k=top_k)
        # 去重：不重复用已经评估过的 doc
        new_hits = [h for h in hits if h[0] not in all_retrieved_ids]
        all_retrieved_ids.update(h[0] for h in hits)
        context = '\n'.join(h[2] for h in (new_hits or hits))

        # 评估
        verdict_obj = evaluate_context(question, context)
        verdict = verdict_obj['verdict']

        trace.append({
            'iter': it + 1,
            'query': current_query,
            'retrieved': [h[0] for h in hits],
            'verdict': verdict,
            'reason': verdict_obj.get('reason', ''),
        })
        if verbose:
            print(f'  iter {it+1}: query={current_query!r} → retrieved {[h[0] for h in hits]} → {verdict} ({verdict_obj.get("reason", "")})')

        if verdict == 'sufficient':
            return {'answer': generate(question, context), 'trace': trace, 'used_iter': it + 1}
        if verdict == 'insufficient' and it == max_iter - 1:
            return {'answer': '根据已有资料，我无法回答这个问题', 'trace': trace, 'used_iter': it + 1}
        # 继续改写
        current_query = rewrite_query(question, context)

    # partial 退路：最后一轮 generate 一下，警示用户答案可能不完整
    final_ctx = '\n'.join(DOCS[i] for i in all_retrieved_ids)
    answer = generate(question, final_ctx)
    return {'answer': '[部分回答] ' + answer, 'trace': trace, 'used_iter': max_iter}

In [ ]:
# 跑 4 个 query：(a) 直接能答 (b) 一次不够要改写 (c) 完全无关
for q in ['XZ4054H 容量是多少', 'XZ4054H 的升级版有什么特性', '今天纽约的天气', 'RAG 系统']:
    print(f'\n=== Q: {q!r} ===')
    result = self_rag(q, max_iter=3, top_k=2, verbose=True)
    print(f'  ➜ 答案 ({result["used_iter"]} 轮): {result["answer"][:120]}')

## 6. Corrective-RAG (CRAG) —— 失败时切换到「外部知识源」

**CRAG 与 Self-RAG 的核心差别**：
- Self-RAG：改写 query 再查同一向量库
- **CRAG：在评估器判 insufficient 时，触发外部 fallback**（如 web search、另一份语料）

**典型 fallback 优先级**：
1. 向量库召回（朴素 RAG）
2. 同一库不同 collection（如 「档案库 → 公开 wiki」）
3. Web search（如 Tavily、Bing）
4. LLM 内部知识（最后一搏，要标注「来源：LLM 知识」）

In [ ]:
def fake_web_search(query: str) -> list[str]:
    """模拟一个 web 兜底源。生产里换成 Tavily / SerpAPI 的 HTTP 调用。"""
    web_kb = {
        '天气':       ['【web】今天北京阴转多云，气温 5–15°C；纽约晴 4–12°C。'],
        '股票':       ['【web】美股三大指数收盘小幅波动。'],
    }
    return [v for k, vs in web_kb.items() for v in vs if k in query]

def crag(question: str, max_iter: int = 2, top_k: int = 2, verbose: bool = True) -> dict:
    # Step 1: 本地向量库
    hits = retrieve(question, top_k=top_k)
    context = '\n'.join(h[2] for h in hits)
    verdict = evaluate_context(question, context)['verdict']
    if verbose:
        print(f'  本地: {[h[0] for h in hits]} → {verdict}')

    if verdict == 'sufficient':
        return {'answer': generate(question, context), 'source': 'local'}

    # Step 2: fallback 到 web
    web_results = fake_web_search(question)
    if web_results:
        web_ctx = '\n'.join(web_results)
        web_verdict = evaluate_context(question, web_ctx)['verdict']
        if verbose:
            print(f'  web : 取到 {len(web_results)} 条 → {web_verdict}')
        if web_verdict in ('sufficient', 'partial'):
            return {'answer': generate(question, web_ctx), 'source': 'web'}

    # Step 3: 没辙了
    return {'answer': '根据已有资料，我无法回答这个问题', 'source': 'none'}

for q in ['XZ4054H 容量', '今天的天气', '量子计算原理']:
    print(f'\n=== Q: {q!r} ===')
    result = crag(q)
    print(f'  ➜ source={result["source"]}  answer={result["answer"][:100]}')

## 深入思考

1. **Self-RAG 论文里有 special tokens（如 `[Retrieve]`、`[Relevant]`），我们这里为什么不用？**
   - 那是为「训练特殊版 SFT 模型」准备的。我们用通用 LLM + prompt + JSON 解析，**收益 80%、成本 20%**。生产里如果有资源，可以照论文训。
2. **`max_iter=3` 经验值，再大不行吗？**
   - 每轮 = 一次 retrieve + 一次 LLM 评估 + 一次 LLM 改写，延迟翻倍。> 3 轮通常收益边际很低，**改 chunking 或加 hybrid 更划算**。
3. **怎么判断「改写后真的更好」？**
   - 加 trace 收集后离线评估。把多轮改写视为一棵「检索树」，**最终选评估分最高的那个 context** 生成（类似 beam search）。
4. **CRAG 的 fallback 顺序怎么定？**
   - 由「**召回率 × 可信度 × 成本**」决定。本地权威库优先 > web 次之 > LLM 内部知识最后（且明确标注）。
5. **多轮循环 vs 一次召回多 (top_k=20) 哪个更好？**
   - 多召回成本低，但召不到就是召不到。多轮能**换角度查**，弥补单次召回的盲区。两者通常组合用。

**改一改**：
- 把 `evaluate_context` 改成更严格（overlap≥6 才算 sufficient），看是否更多触发改写
- 加一个「LLM 答案自检」步骤：生成后再让 LLM 评估「这个答案是否基于了 context」（faithfulness 自检），见 notebook 24

## 自检 ✅

- [ ] 画出 Self-RAG 的循环图（4 个节点 + 边）。
- [ ] 解释 Self-RAG 与 Corrective-RAG 的核心差异。
- [ ] 写出一个 LLM 评估 prompt 的 3 个必要约束（输出格式 / 选项枚举 / 不要解释）。
- [ ] 解释「为什么必须设 max_iter」。
- [ ] 给一个 RAG 产品反馈「有时回答完全偏题」，能立刻问「有没有上 self-eval？」

## 下一步

进入 Stage 4 → [`../stage4_专家/24_eval_and_serving.ipynb`](../stage4_专家/24_eval_and_serving.ipynb)